
# People Analytics — Analyse complète de l'attrition

## Table des matières
1. [Partie 1 — Analyse Exploratoire (EDA)](#partie-1)
2. [Partie 2 — KPI & Analyse RH](#partie-2)
3. [Partie 3 — Machine Learning](#partie-3)
4. [Conclusion générale](#conclusion)

---
Ce notebook présente une démarche **People Analytics** de bout en bout, orientée **décision RH** et **prévention proactive du turnover**.


In [ ]:

# Imports & setup
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

# Dépendances optionnelles
XGB_AVAILABLE = True
SHAP_AVAILABLE = True
try:
    from xgboost import XGBClassifier
except Exception:
    XGB_AVAILABLE = False

try:
    import shap
except Exception:
    SHAP_AVAILABLE = False

# Style visuel professionnel
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

ATTRITION_PALETTE = {0: '#2E8B57', 1: '#C0392B'}
SEQUENTIAL_PALETTE = 'viridis'


def load_dataset():
    candidates = ['./people_analytics_dataset.csv', '/content/people_analytics_dataset.csv']
    for path in candidates:
        if os.path.exists(path):
            print(f"Dataset chargé depuis: {path}")
            return pd.read_csv(path)
    raise FileNotFoundError('Impossible de localiser people_analytics_dataset.csv')


def attrition_rate_table(df, col):
    table = df.groupby(col, dropna=False)['attrition'].mean().mul(100).sort_values(ascending=False)
    return table.reset_index(name='attrition_rate_pct')


def plot_attrition_rate_by(df, col, top_n=None, title=None):
    rate_df = attrition_rate_table(df, col)
    if top_n is not None:
        rate_df = rate_df.head(top_n)
    plt.figure(figsize=(11, 5))
    ax = sns.barplot(
        data=rate_df,
        x=col,
        y='attrition_rate_pct',
        hue=col,
        palette=SEQUENTIAL_PALETTE,
        legend=False
    )
    global_rate = df['attrition'].mean() * 100
    ax.axhline(global_rate, color='red', linestyle='--', linewidth=1.5, label=f'Moyenne globale: {global_rate:.2f}%')
    ax.set_title(title if title else f"**Taux d\'attrition par {col}**")
    ax.set_xlabel(col)
    ax.set_ylabel("Taux d\'attrition (%)")
    ax.tick_params(axis='x', rotation=35)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()


def label_engagement(score):
    if score < 40:
        return 'Faible'
    if score < 70:
        return 'Moyen'
    return 'Élevé'


In [ ]:

# Chargement des données
raw_df = load_dataset()
df = raw_df.copy()



<a id='partie-1'></a>
# Partie 1 — Analyse Exploratoire (EDA)


In [ ]:

# Vue d'ensemble
display(pd.DataFrame({'lignes': [df.shape[0]], 'colonnes': [df.shape[1]]}))
display(df.head())
display(df.dtypes.to_frame('dtype').T)
display(df.describe(include='all').T.head(15))


In [ ]:

# Qualité des données : valeurs manquantes
missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['variable', 'pct_manquant']

print('Variables avec valeurs manquantes:')
display(missing_df)

if not missing_df.empty:
    plt.figure(figsize=(9, 4))
    ax = sns.barplot(data=missing_df, x='variable', y='pct_manquant', hue='variable', palette='magma', legend=False)
    ax.set_title('**Pourcentage de valeurs manquantes par variable**')
    ax.set_xlabel('Variable')
    ax.set_ylabel('% manquant')
    ax.tick_params(axis='x', rotation=25)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(11, 4))
sns.heatmap(df.isna(), cbar=False)
plt.title('**Heatmap des valeurs manquantes**')
plt.xlabel('Variables')
plt.ylabel('Observations')
plt.tight_layout()
plt.show()


In [ ]:

# Doublons exacts
n_dups = df.duplicated().sum()
print(f'Nombre de doublons exacts avant traitement: {n_dups}')

dups_preview = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(20)
print("Aperçu des doublons (jusqu'à 20 lignes):")
display(dups_preview)

df = df.drop_duplicates().reset_index(drop=True)
print(f'Nombre de doublons après traitement: {df.duplicated().sum()}')
print(f'Nouvelle dimension: {df.shape}')


In [ ]:

# Outliers sur variables clés (IQR, sans suppression aveugle)
key_vars = ['salary', 'overtime_hours', 'absenteeism_days']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(key_vars):
    sns.boxplot(data=df, y=col, ax=axes[i], color='#4C72B0')
    axes[i].set_title(f'**Boxplot — {col}**')
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    axes[i].set_xlabel(f'Outliers IQR: {n_out}')
plt.tight_layout()
plt.show()


In [ ]:

# Distribution de la cible attrition
attr_counts = df['attrition'].value_counts().sort_index()
attr_pct = df['attrition'].value_counts(normalize=True).sort_index().mul(100)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='attrition', hue='attrition', palette=ATTRITION_PALETTE, legend=False)
ax.set_title('**Distribution de la cible attrition**')
ax.set_xlabel('Attrition (0=reste, 1=quitte)')
ax.set_ylabel('Nombre de collaborateurs')
for i, p in enumerate(ax.patches):
    ax.annotate(f"{attr_pct.iloc[i]:.2f}%", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print(f"Taux d'attrition observé: {attr_pct.loc[1]:.2f}% -> Classe positive fortement déséquilibrée.")


In [ ]:

# Analyse univariée (variables numériques clés)
num_cols = ['age', 'salary', 'years_at_company', 'overtime_hours', 'engagement_score',
            'psychological_safety_score', 'manager_favoritism_score', 'visibility_score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='#4C72B0')
    ax.set_title(f'**Distribution de {col}**')
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()


In [ ]:

# Analyse catégorielle
cat_cols = ['department', 'job_role', 'country', 'gender', 'education_level', 'school_tier', 'parental_status']

for col in cat_cols:
    plt.figure(figsize=(11, 4))
    order = df[col].value_counts().index
    ax = sns.countplot(data=df, x=col, order=order, hue=col, palette=SEQUENTIAL_PALETTE, legend=False)
    ax.set_title(f'**Répartition des effectifs — {col}**')
    ax.set_xlabel(col)
    ax.set_ylabel('Effectif')
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.show()


In [ ]:

# Analyse bivariée : numériques vs attrition
bivar_num_cols = ['salary', 'overtime_hours', 'years_at_company', 'engagement_score',
                  'psychological_safety_score', 'manager_favoritism_score', 'visibility_score']

for col in bivar_num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=df, x='attrition', y=col, hue='attrition', palette=ATTRITION_PALETTE, legend=False, ax=axes[0])
    axes[0].set_title(f'**Boxplot {col} vs attrition**')
    sns.violinplot(data=df, x='attrition', y=col, hue='attrition', palette=ATTRITION_PALETTE, legend=False, ax=axes[1], cut=0)
    axes[1].set_title(f'**Violinplot {col} vs attrition**')
    for ax in axes:
        ax.set_xlabel('Attrition')
    plt.tight_layout()
    plt.show()


In [ ]:

# Analyse bivariée : taux d'attrition par catégories
for col in cat_cols:
    plot_attrition_rate_by(df, col, title=f"**Taux de turnover (%) par {col}**")


In [ ]:

# Corrélations
numeric_df = df.select_dtypes(include=[np.number]).copy()
corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
annot = corr.copy().round(2).astype(str)
annot = annot.where(corr.abs() > 0.10, '')

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    mask=mask,
    cmap='coolwarm',
    center=0,
    annot=annot,
    fmt='',
    linewidths=0.5,
    cbar_kws={'shrink': 0.7}
)
plt.title('**Matrice de corrélation (annotations pour |r| > 0.10)**')
plt.tight_layout()
plt.show()

top_attr_corr = corr['attrition'].drop('attrition').sort_values(key=np.abs, ascending=False).head(10)
print('Top corrélations (absolues) avec attrition:')
display(top_attr_corr.to_frame('corr_attrition'))



### Interprétation business — Partie 1 (EDA)
- Le taux d'attrition (~2.6%) confirme un **déséquilibre de classes sévère**, ce qui impose des métriques adaptées (Recall, PR-AUC) en ML.
- Les variables liées à la **charge de travail** (`overtime_hours`, `absenteeism_days`) et au **climat managérial** (`psychological_safety_score`, `manager_favoritism_score`) apparaissent comme des pistes RH prioritaires.
- Les écarts de turnover par métier/pays/département suggèrent une hétérogénéité organisationnelle: les actions RH devront être **ciblées par population** et non généralisées.
- Les 20 doublons ont été supprimés pour sécuriser la qualité analytique et éviter un biais dans les indicateurs.



<a id='partie-2'></a>
# Partie 2 — KPI & Analyse RH


In [ ]:

# Préparation des variables KPI
kpi_df = df.copy()
kpi_df['years_bucket'] = pd.cut(
    kpi_df['years_at_company'],
    bins=[-np.inf, 1, 3, 6, 10, np.inf],
    labels=['0-1', '2-3', '4-6', '7-10', '10+']
)

# A. KPI Turnover global
turnover_global = kpi_df['attrition'].mean() * 100

plt.figure(figsize=(8, 2.5))
plt.axis('off')
plt.text(0.5, 0.55, f"{turnover_global:.2f}%", ha='center', va='center', fontsize=36, color='#C0392B', fontweight='bold')
plt.text(0.5, 0.20, "Taux global de turnover", ha='center', va='center', fontsize=14)
plt.title('**KPI principal — Turnover global**')
plt.tight_layout()
plt.show()


In [ ]:

# A. Turnover par dimensions RH
plot_attrition_rate_by(kpi_df, 'department', title='**Turnover par département**')
plot_attrition_rate_by(kpi_df, 'job_role', top_n=12, title='**Turnover par job role (top 12)**')
plot_attrition_rate_by(kpi_df, 'years_bucket', title='**Turnover par ancienneté (buckets)**')
plot_attrition_rate_by(kpi_df, 'gender', title='**Turnover par genre**')
plot_attrition_rate_by(kpi_df, 'manager_level', title='**Turnover par niveau hiérarchique**')
plot_attrition_rate_by(kpi_df, 'country', title='**Turnover par pays**')


In [ ]:

# B. Management & culture
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.violinplot(data=kpi_df, x='attrition', y='manager_favoritism_score', hue='attrition', palette=ATTRITION_PALETTE, legend=False, cut=0, ax=axes[0])
axes[0].set_title('**Favoritisme managérial vs attrition**')
sns.violinplot(data=kpi_df, x='attrition', y='psychological_safety_score', hue='attrition', palette=ATTRITION_PALETTE, legend=False, cut=0, ax=axes[1])
axes[1].set_title('**Sécurité psychologique vs attrition**')
plt.tight_layout()
plt.show()

risk_table = (
    kpi_df.groupby(['department', 'job_role'])
    .agg(attrition_rate=('attrition', 'mean'), psych_safety_mean=('psychological_safety_score', 'mean'))
    .reset_index()
)
risk_table['attrition_rate_pct'] = risk_table['attrition_rate'] * 100
risk_table = risk_table.sort_values(['attrition_rate_pct', 'psych_safety_mean'], ascending=[False, True])
print('Départements / rôles à risque (top 15):')
display(risk_table.head(15)[['department', 'job_role', 'attrition_rate_pct', 'psych_safety_mean']])

fav_promo = (
    kpi_df.groupby('promotion_last_3y')
    .agg(favoritism_mean=('manager_favoritism_score', 'mean'), attrition_rate=('attrition', 'mean'))
    .reset_index()
)
fav_promo['attrition_rate_pct'] = fav_promo['attrition_rate'] * 100
display(fav_promo)


In [ ]:

# C. Burn-out / Bore-out
kpi_df['workload_risk'] = (
    ((kpi_df['overtime_hours'] - kpi_df['overtime_hours'].mean()) / kpi_df['overtime_hours'].std()) +
    ((kpi_df['absenteeism_days'] - kpi_df['absenteeism_days'].mean()) / kpi_df['absenteeism_days'].std())
) / 2

kpi_df['workload_risk_level'] = pd.qcut(kpi_df['workload_risk'], q=3, labels=['Faible', 'Moyen', 'Élevé'])

plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=kpi_df,
    x='overtime_hours',
    y='psychological_safety_score',
    hue='attrition',
    palette=ATTRITION_PALETTE,
    alpha=0.5
)
ax.axvline(kpi_df['overtime_hours'].median(), color='black', linestyle='--', linewidth=1)
ax.axhline(kpi_df['psychological_safety_score'].median(), color='black', linestyle='--', linewidth=1)
ax.set_title('**Overtime vs sécurité psychologique (coloré par attrition)**')
ax.set_xlabel('Heures supplémentaires')
ax.set_ylabel('Score de sécurité psychologique')
plt.tight_layout()
plt.show()

kpi_df['engagement_level'] = kpi_df['engagement_score'].apply(label_engagement)
kpi_df['overtime_band'] = pd.qcut(kpi_df['overtime_hours'], q=3, labels=['Charge faible', 'Charge moyenne', 'Charge élevée'])

burn_bore = pd.crosstab(kpi_df['engagement_level'], kpi_df['overtime_band'], normalize='columns').mul(100)
print('Matrice engagement x charge (% colonne):')
display(burn_bore)

plt.figure(figsize=(8, 4))
sns.heatmap(burn_bore, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title('**Engagement vs charge de travail (burn-out / bore-out)**')
plt.xlabel('Niveau de charge (overtime)')
plt.ylabel("Niveau d'engagement")
plt.tight_layout()
plt.show()


In [ ]:

# D. Équité (rémunération, promotion, mobilité)
kpi_df['salary_band'] = pd.qcut(kpi_df['salary'], q=5, labels=['Band 1', 'Band 2', 'Band 3', 'Band 4', 'Band 5'])

salary_gender_attr = (
    kpi_df.groupby(['salary_band', 'gender'])['attrition'].mean().mul(100).reset_index(name='attrition_rate_pct')
)

plt.figure(figsize=(9, 4))
ax = sns.barplot(data=salary_gender_attr, x='salary_band', y='attrition_rate_pct', hue='gender', palette='Set2')
ax.axhline(turnover_global, color='red', linestyle='--', linewidth=1.5, label='Moyenne globale')
ax.set_title('**Attrition par bande salariale et genre**')
ax.set_xlabel('Bande salariale')
ax.set_ylabel("Taux d\'attrition (%)")
ax.legend()
plt.tight_layout()
plt.show()

salary_heat = kpi_df.pivot_table(index='job_role', columns='gender', values='salary', aggfunc='median')
plt.figure(figsize=(8, 7))
sns.heatmap(salary_heat, annot=True, fmt='.0f', cmap='Blues')
plt.title('**Salaire médian par genre et job role**')
plt.xlabel('Genre')
plt.ylabel('Job role')
plt.tight_layout()
plt.show()

promo_mob = kpi_df.groupby('attrition').agg(
    promotion_rate=('promotion_last_3y', 'mean'),
    internal_mobility_mean=('internal_mobility_count', 'mean')
)
promo_mob['promotion_rate_pct'] = promo_mob['promotion_rate'] * 100
print('Promotion et mobilité selon attrition:')
display(promo_mob)



### Interprétation business — Partie 2 (KPI & RH)
1. Les périmètres présentant un **turnover supérieur à la moyenne globale** doivent être priorisés dans le plan RH trimestriel.
2. Les couples *département × job role* cumulant **attrition élevée** et **faible sécurité psychologique** sont des zones critiques: audit managérial ciblé recommandé.
3. Les signaux de **burn-out** (fort overtime + faible engagement) et de **bore-out** (faible charge + faible engagement) coexistent: il faut piloter charge et sens du travail simultanément.
4. Les écarts observés en rémunération/promotion/mobilité justifient un suivi d'**équité RH** par segment (genre, rôle, niveau).
5. Recommandation opérationnelle: mettre sous surveillance les équipes dont le score moyen de sécurité psychologique est inférieur à 55, avec coaching managers et revue des pratiques d'évolution.



<a id='partie-3'></a>
# Partie 3 — Machine Learning


In [ ]:

# Préparation ML
ml_df = raw_df.copy().drop_duplicates().reset_index(drop=True)

# Imputation médiane demandée
for col in ['performance_rating', 'engagement_score', 'salary']:
    ml_df[col] = ml_df[col].fillna(ml_df[col].median())

# Variables
TARGET = 'attrition'
ID_COL = 'employee_id'
cat_features = ['gender', 'country', 'department', 'job_role', 'education_level', 'school_tier', 'parental_status']

X = ml_df.drop(columns=[TARGET, ID_COL])
y = ml_df[TARGET]
num_features = [c for c in X.columns if c not in cat_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print('Train/Test sizes:', X_train.shape, X_test.shape)
print('Taux attrition train:', y_train.mean().round(4), '| test:', y_test.mean().round(4))

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ]
)

class_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'scale_pos_weight estimé: {class_ratio:.2f}')


In [ ]:

# Modèles et stratégies de déséquilibre (class_weight vs SMOTE)
model_candidates = {
    'Logistic Regression': {
        'class_weight': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced'),
        'smote': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    },
    'Random Forest': {
        'class_weight': RandomForestClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            class_weight='balanced',
            n_jobs=-1
        ),
        'smote': RandomForestClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }
}

if XGB_AVAILABLE:
    model_candidates['XGBoost'] = {
        'class_weight': XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            scale_pos_weight=class_ratio,
            n_jobs=-1
        ),
        'smote': XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }
else:
    # Fallback gracieux si XGBoost indisponible
    model_candidates['Gradient Boosting (fallback)'] = {
        'class_weight': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'smote': GradientBoostingClassifier(random_state=RANDOM_STATE)
    }

results = []
trained_models = {}
curve_data = {}

for model_name, strategies in model_candidates.items():
    for strategy, estimator in strategies.items():
        run_name = f"{model_name} + {strategy}"
        if strategy == 'smote':
            pipe = ImbPipeline(steps=[
                ('preprocess', preprocessor),
                ('smote', SMOTE(random_state=RANDOM_STATE)),
                ('model', estimator)
            ])
        else:
            pipe = Pipeline(steps=[
                ('preprocess', preprocessor),
                ('model', estimator)
            ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        if hasattr(pipe, 'predict_proba'):
            y_score = pipe.predict_proba(X_test)[:, 1]
        else:
            y_score = pipe.decision_function(X_test)

        metrics = {
            'model': run_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, y_score),
            'PR-AUC': average_precision_score(y_test, y_score)
        }
        results.append(metrics)
        trained_models[run_name] = pipe

        fpr, tpr, _ = roc_curve(y_test, y_score)
        prec, rec, _ = precision_recall_curve(y_test, y_score)
        curve_data[run_name] = {'fpr': fpr, 'tpr': tpr, 'precision': prec, 'recall': rec, 'score': y_score}

results_df = pd.DataFrame(results).sort_values(by='PR-AUC', ascending=False)
display(results_df.round(4))


In [ ]:

# Visualisation des performances (ROC + Precision-Recall)
plt.figure(figsize=(10, 6))
for model_name, data in curve_data.items():
    plt.plot(data['fpr'], data['tpr'], label=model_name)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('**Courbes ROC des modèles**')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
for model_name, data in curve_data.items():
    plt.plot(data['recall'], data['precision'], label=model_name)
base_rate = y_test.mean()
plt.axhline(base_rate, linestyle='--', color='gray', label=f'Baseline PR: {base_rate:.3f}')
plt.title('**Courbes Precision-Recall (pertinentes en classes déséquilibrées)**')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['model']
best_model = trained_models[best_model_name]
print('Meilleur modèle (tri PR-AUC):', best_model_name)

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
plt.title(f'**Matrice de confusion — {best_model_name}**')
plt.tight_layout()
plt.show()



### Interprétation métier des métriques
- **Accuracy**: part totale des prédictions correctes, mais potentiellement trompeuse quand la classe "départ" est rare.
- **Precision**: parmi les salariés détectés à risque, part réellement en départ (coût des faux positifs).
- **Recall**: parmi les départs réels, part détectée par le modèle (**clé** pour anticiper le turnover).
- **F1-score**: compromis Precision/Recall, utile quand on veut équilibrer détection et fiabilité.
- **ROC-AUC**: capacité globale de séparation entre classes.
- **PR-AUC**: métrique la plus informative ici, car focalisée sur la classe minoritaire (attrition = 1).


In [ ]:

# Interprétabilité : feature importance (modèles arbre)
feature_names = trained_models[best_model_name].named_steps['preprocess'].get_feature_names_out()

# Choisir le meilleur modèle "arbre" pour l'interprétation
tree_candidates = [
    m for m in results_df['model']
    if ('Random Forest' in m) or ('XGBoost' in m) or ('Gradient Boosting' in m)
]

if tree_candidates:
    best_tree_name = tree_candidates[0]
    best_tree_model = trained_models[best_tree_name]
    estimator = best_tree_model.named_steps['model']

    if hasattr(estimator, 'feature_importances_'):
        imp_df = pd.DataFrame({
            'feature': feature_names,
            'importance': estimator.feature_importances_
        }).sort_values('importance', ascending=False).head(15)

        plt.figure(figsize=(10, 6))
        ax = sns.barplot(data=imp_df, x='importance', y='feature', hue='feature', palette='viridis', legend=False)
        ax.set_title(f'**Top 15 variables importantes — {best_tree_name}**')
        ax.set_xlabel('Importance')
        ax.set_ylabel('Variable')
        plt.tight_layout()
        plt.show()
    else:
        print('Le modèle arbre sélectionné ne fournit pas de feature_importances_.')
else:
    print('Aucun modèle arbre disponible pour la feature importance.')


In [ ]:

# SHAP (fallback gracieux)
if SHAP_AVAILABLE and tree_candidates:
    try:
        best_tree_name = tree_candidates[0]
        best_tree_model = trained_models[best_tree_name]
        fitted_preprocessor = best_tree_model.named_steps['preprocess']
        fitted_estimator = best_tree_model.named_steps['model']

        X_test_t = fitted_preprocessor.transform(X_test)
        feature_names = fitted_preprocessor.get_feature_names_out()

        sample_size = min(1000, X_test_t.shape[0])
        X_sample_t = X_test_t[:sample_size]
        X_sample_df = pd.DataFrame(X_sample_t, columns=feature_names)

        explainer = shap.TreeExplainer(fitted_estimator)
        shap_values = explainer.shap_values(X_sample_df)

        if isinstance(shap_values, list):
            sv = shap_values[1] if len(shap_values) > 1 else shap_values[0]
        elif hasattr(shap_values, 'values'):
            sv = shap_values.values
            if sv.ndim == 3:
                sv = sv[:, :, 1]
        else:
            sv = shap_values

        plt.figure()
        shap.summary_plot(sv, X_sample_df, show=False)
        plt.title(f'SHAP summary plot — {best_tree_name}')
        plt.tight_layout()
        plt.show()

        for col in ['num__overtime_hours', 'num__psychological_safety_score', 'num__manager_favoritism_score']:
            if col in X_sample_df.columns:
                shap.dependence_plot(col, sv, X_sample_df, show=False)
                plt.title(f'SHAP dependence — {col}')
                plt.tight_layout()
                plt.show()
    except Exception as e:
        print(f"SHAP indisponible ou erreur d'exécution: {e}")
else:
    print('SHAP non exécuté (librairie indisponible ou modèle non compatible).')



### Analyse critique (obligatoire)
- Le déséquilibre de classes reste la principale difficulté: la performance doit être pilotée en priorité via **Recall/PR-AUC**, pas via l'accuracy seule.
- Les données contiennent des variables subjectives (scores managériaux/engagement) potentiellement sensibles au biais de mesure.
- Risque de surapprentissage: à surveiller via validation croisée et stabilité temporelle avant déploiement.
- Limites éthiques et conformité: variables sensibles (`gender`, `country`, `parental_status`, `accented_name_flag`) exposent à un risque de discrimination algorithmique.
- Recommandation forte: usage du modèle comme **outil d'aide à la décision collective**, jamais comme moteur de décision individuelle automatisée (RGPD, humain dans la boucle).



<a id='conclusion'></a>
# Conclusion générale

Ce livrable suit une chaîne complète **donnée brute → diagnostic RH → modélisation prédictive → recommandations**.

- **Partie 1 (EDA)** : sécurisation de la qualité des données, identification de signaux d'attrition, confirmation du déséquilibre de classes.
- **Partie 2 (KPI RH)** : priorisation des populations à risque (départements/rôles), lecture management/culture, signaux de burn-out et d'équité.
- **Partie 3 (ML)** : comparaison de modèles robustes avec gestion du déséquilibre, évaluation adaptée, interprétabilité (importance + SHAP si disponible).

### Roadmap RH actionnable (90 jours)
1. Cibler les équipes au-dessus du turnover moyen et sous le seuil de sécurité psychologique.
2. Lancer un audit d'équité (rémunération, promotion, mobilité) par rôle et genre.
3. Mettre en place un dispositif de prévention (charge de travail, engagement, coaching managers).
4. Industrialiser le modèle en mode pilote avec gouvernance éthique et revue humaine systématique.
